In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 数据录入 (Data Entry)
# ==========================================

# 训练集特征 (X_train) - 来自表 8.7
# 顺序：氯, 硫化氢, 二氧化硫, 碳四, 环氧氯丙烷, 环己烷
X_train = np.array([
    [0.0560, 0.0840, 0.0310, 0.0380, 0.0081, 0.0220], # 样本1
    [0.0400, 0.0550, 0.1000, 0.1100, 0.0220, 0.0073], # 样本2
    [0.0500, 0.0740, 0.0410, 0.0480, 0.0071, 0.0200], # 样本3
    [0.0450, 0.0500, 0.1100, 0.1000, 0.0250, 0.0063], # 样本4
    [0.0380, 0.1300, 0.0790, 0.1700, 0.0580, 0.0430], # 样本5
    [0.0300, 0.1100, 0.0700, 0.1600, 0.0500, 0.0460], # 样本6
    [0.0340, 0.0950, 0.0580, 0.1600, 0.2000, 0.0290], # 样本7
    [0.0300, 0.0900, 0.0680, 0.1800, 0.2200, 0.0390], # 样本8
    [0.0840, 0.0660, 0.0290, 0.3200, 0.0120, 0.0410], # 样本9
    [0.0850, 0.0760, 0.0190, 0.3000, 0.0100, 0.0400], # 样本10
    [0.0640, 0.0720, 0.0200, 0.2500, 0.0280, 0.0380], # 样本11
    [0.0540, 0.0650, 0.0220, 0.2800, 0.0210, 0.0400], # 样本12
    [0.0480, 0.0890, 0.0620, 0.2600, 0.0380, 0.0360], # 样本13
    [0.0450, 0.0920, 0.0720, 0.2000, 0.0350, 0.0320], # 样本14
    [0.0690, 0.0870, 0.0270, 0.0500, 0.0890, 0.0210]  # 样本15
])

# 训练集标签 (y_train) - 来自表 8.7 (污染类别)
y_train = np.array([1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 1])

# 待判断样本 (X_test) - 来自表 8.8
X_test = np.array([
    [0.0520, 0.0840, 0.0210, 0.0370, 0.0071, 0.0220], # 待判1
    [0.0410, 0.0550, 0.1100, 0.1100, 0.0210, 0.0073], # 待判2
    [0.0300, 0.1120, 0.0720, 0.1600, 0.0560, 0.0210], # 待判3
    [0.0740, 0.0830, 0.1050, 0.1900, 0.0200, 1.0000]  # 待判4
])

# ==========================================
# 2. 模型训练与预测 (Training & Prediction)
# ==========================================

# --- 预处理：标准化 (Standardization) ---
# KNN 算法基于距离，特征值的量纲差异会影响结果，因此需要标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 算法 1：K-近邻 (KNN) ---
# 选择最近的 3 个邻居进行投票
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# --- 算法 2：决策树 (Decision Tree) ---
# 决策树通常不需要数据标准化
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# ==========================================
# 3. 输出结果 (Output)
# ==========================================
print("实验结果：")
print("-" * 50)
print(f"{'样本ID':<8} | {'KNN 预测类别':<15} | {'决策树 预测类别':<15}")
print("-" * 50)

for i in range(len(X_test)):
    print(f"样本 {i+1:<5} | {y_pred_knn[i]:<19} | {y_pred_dt[i]:<19}")
print("-" * 50)

df_result = pd.DataFrame({
    '待判样本': [1, 2, 3, 4],
    'KNN结果': y_pred_knn,
    '决策树结果': y_pred_dt
})

实验结果：
--------------------------------------------------
样本ID     | KNN 预测类别        | 决策树 预测类别       
--------------------------------------------------
样本 1     | 1                   | 1                  
样本 2     | 1                   | 1                  
样本 3     | 2                   | 1                  
样本 4     | 2                   | 2                  
--------------------------------------------------


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# ==========================================
# 1. 数据录入 (Data Entry)
# ==========================================
# 将图片中的表格数据转化为 DataFrame
data = {
    'Country': [
        '中国', '美国', '日本', '德国', '英国', '法国', '意大利', 
        '加拿大', '澳大利亚', '俄罗斯', '捷克', '波兰', '匈牙利', 
        '南斯拉夫', '罗马尼亚', '保加利亚', '印度', '印度尼西亚', 
        '尼日利亚', '墨西哥', '巴西'
    ],
    '森林面积': [
        11978, 28446, 2501, 1028, 210, 1458, 635, 
        32613, 10700, 92000, 458, 868, 161, 
        929, 634, 385, 6748, 2180, 
        1490, 4850, 57500
    ],
    '森林覆盖率': [
        12.5, 30.4, 67.2, 28.4, 8.6, 26.7, 21.1, 
        32.7, 13.9, 41.1, 35.8, 27.8, 17.4, 
        36.3, 26.7, 34.7, 20.5, 84.0, 
        16.1, 24.6, 67.6
    ],
    '林木蓄积量': [
        93.5, 202.0, 24.8, 14.0, 1.5, 16.0, 3.6, 
        192.8, 10.5, 841.5, 8.9, 11.4, 2.5, 
        11.4, 11.3, 2.5, 29.0, 33.7, 
        0.8, 32.6, 238.0
    ],
    '草原面积': [
        31908, 23754, 58, 599, 1147, 1288, 514, 
        2385, 45190, 37370, 168, 405, 129, 
        640, 447, 200, 1200, 1200, 
        2090, 7450, 15900
    ]
}

df = pd.DataFrame(data)

# ==========================================
# 2. 数据预处理 (Preprocessing)
# ==========================================
# 提取数值列进行计算
features = ['森林面积', '森林覆盖率', '林木蓄积量', '草原面积']
X = df[features]

# 数据标准化 (Standardization) - 非常重要！
# 因为“森林面积”是几万，“覆盖率”是几十，不标准化会导致结果偏差。
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# 3. 聚类模型训练 (K-Means Clustering)
# ==========================================
# 这里我们将国家分为 4 类 (K=4)，你可以尝试改成 3 或 5 查看不同效果
kmeans = KMeans(n_clusters=4, random_state=42)
df['聚类结果'] = kmeans.fit_predict(X_scaled)

# ==========================================
# 4. 输出结果分析 (Result Analysis)
# ==========================================
print("【聚类分析结果】")
print("-" * 50)

# 按类别分组打印
for cluster_id in sorted(df['聚类结果'].unique()):
    print(f"\n>>> 类别 {cluster_id + 1} 包含的国家:")
    countries = df[df['聚类结果'] == cluster_id]['Country'].values
    print(", ".join(countries))
    
    # 计算该类的平均特征，帮助理解这一类的特点
    mean_stats = df[df['聚类结果'] == cluster_id][features].mean()
    print(f"   (特征描述: 森林面积约 {mean_stats['森林面积']:.0f}, 覆盖率约 {mean_stats['森林覆盖率']:.1f}%, 草原面积约 {mean_stats['草原面积']:.0f})")



【聚类分析结果】
--------------------------------------------------

>>> 类别 1 包含的国家:
加拿大, 巴西
   (特征描述: 森林面积约 45056, 覆盖率约 50.1%, 草原面积约 9142)

>>> 类别 2 包含的国家:
日本, 德国, 英国, 法国, 意大利, 捷克, 波兰, 匈牙利, 南斯拉夫, 罗马尼亚, 保加利亚, 印度, 印度尼西亚, 尼日利亚, 墨西哥
   (特征描述: 森林面积约 1636, 覆盖率约 31.7%, 草原面积约 1169)

>>> 类别 3 包含的国家:
中国, 美国, 澳大利亚
   (特征描述: 森林面积约 17041, 覆盖率约 18.9%, 草原面积约 33617)

>>> 类别 4 包含的国家:
俄罗斯
   (特征描述: 森林面积约 92000, 覆盖率约 41.1%, 草原面积约 37370)


d:\autumn\My_Envir\project\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [4]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. 加载数据
# UCI Adult 数据集通常没有表头，我们需要手动定义
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", 
    "marital-status", "occupation", "relationship", "race", "sex", 
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "salary"
]

# 读取数据，注意处理原始数据中的空格
print("正在从 UCI 下载并读取数据...")
df = pd.read_csv(url, names=column_names, skipinitialspace=True)

# 2. 数据预处理
# 筛选收入大于 50k 的数据
df_high_income = df[df['salary'] == '>50K'].copy()

print(f"收入 >50K 的样本数量: {len(df_high_income)}")

# 筛选题目指定的7个变量
target_cols = [
    'workclass', 'education', 'marital-status', 'occupation', 
    'relationship', 'race', 'sex'
]
df_subset = df_high_income[target_cols]

# 3. 数据转换 (One-Hot Encoding)
# Apriori 算法需要 0/1 矩阵或布尔值矩阵
# pd.get_dummies 会把 'sex' 变成 'sex_Male', 'sex_Female' 等列
df_ohe = pd.get_dummies(df_subset)

# 为了节省内存和提高可读性，将所有值转换为布尔类型 (True/False)
df_ohe = df_ohe.astype(bool)

# 4. 挖掘关联规则
print("正在挖掘频繁项集...")
# min_support: 最小支持度。设为 0.1 意味着该组合至少要在 10% 的数据中出现
# 如果结果太少，可以降低这个值；如果太慢，可以提高这个值
frequent_itemsets = apriori(df_ohe, min_support=0.15, use_colnames=True)

print("正在生成关联规则...")
# metric="lift": 使用提升度作为评估标准 (也可以用 confidence)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 5. 提取前 10 条规则
# 我们按 'lift' (提升度) 降序排列，提升度越高表示关联性越强（不仅仅是巧合）
top_10_rules = rules.sort_values(by='lift', ascending=False).head(10)

# 6. 格式化输出
print("\n========== 关联性排名前 10 的规则 (针对 >50K 群体) ==========")
# 重新组织列以便阅读
output_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print(top_10_rules[output_cols].to_string(index=False))

# 简单的解释辅助
print("\n========== 结果解读示例 ==========")
if not top_10_rules.empty:
    first_rule = top_10_rules.iloc[0]
    ant = list(first_rule['antecedents'])[0]
    con = list(first_rule['consequents'])[0]
    print(f"第一条规则意味着：在收入>50K的人群中，如果是 '{ant}'，")
    print(f"那么大概率也是 '{con}'。")
    print(f"Lift (提升度) 为 {first_rule['lift']:.2f}，表示这种关联比随机猜测强 {first_rule['lift']:.2f} 倍。")

正在从 UCI 下载并读取数据...
收入 >50K 的样本数量: 7841
正在挖掘频繁项集...
正在生成关联规则...

========== 关联性排名前 10 的规则 (针对 >50K 群体) ==========
                                                              antecedents                                                               consequents  support  confidence     lift
                (sex_Male, race_White, marital-status_Married-civ-spouse)                        (occupation_Exec-managerial, relationship_Husband) 0.178804    0.257531 1.342621
                       (occupation_Exec-managerial, relationship_Husband)                 (sex_Male, race_White, marital-status_Married-civ-spouse) 0.178804    0.932181 1.342621
(sex_Male, occupation_Exec-managerial, marital-status_Married-civ-spouse)                                        (race_White, relationship_Husband) 0.178804    0.927862 1.340341
                                       (race_White, relationship_Husband) (sex_Male, occupation_Exec-managerial, marital-status_Married-civ-spouse) 0.178804    0.258290 1.3403